# Lesson 92 · Six months of modules join up for the first time — the end-to-end pipeline: microphone → ASR → LLM → answer

**Learning goals**
1. Understand how end-to-end latency breaks down into three segments: ASR / RAG / LLM.
2. Learn to handle stage failures with fallbacks and a three-field result interface.
3. See how asyncio concurrency brings down perceived latency.
4. Be able to locate a pipeline bottleneck with logs and metrics.

> Look at latency and throughput first, then at how the chain is put together; microphone → ASR → LLM join up here for the first time.

**Goal**: chain together everything Aurora has built month by month — live recording → Whisper transcription → Podcast Agent answer — and build intuition for system integration and how latency is distributed.

🔗 Aurora connection: `aurora.audio.io` (recording / reading and writing, implemented), `aurora.llm.retrieve` (TF-IDF retrieval, implemented), `aurora.llm.sample` (sampling / decoding, implemented); `aurora.speech` (ASR, planned), `aurora.rag` (standalone retrieval core, planned)

> **Note**: this lesson demonstrates the integration pattern with mock functions; once the speech/rag modules are ready, you can swap the mocks out one at a time for the real implementations.

← **Previous lesson** · [L91 · Attention illustrated](../9_llm/L91_visual_llm.ipynb)

> Last lesson covered **attention illustrated**: heatmaps of single-head self-attention weights, and a visualization of LoRA's low-rank structure.
> This lesson explores **the end-to-end pipeline**.

The core question of this lesson: **when you chain independent modules together, where is the bottleneck?**
End-to-end latency is not the maximum of the stages — it is their sum; and a failure at any link can bring the whole chain down.
So every stage needs timing and a fallback, rather than an assumption that everything will be fine.

## 🎯 Every link in this chain is one you built yourself

Integration isn't new knowledge — it takes the cliffs of understanding you climbed one by one over six months and, for the first time, joins them into a single chain:

| Pipeline stage | Which lessons it comes from | The core you've already implemented yourself |
|---|---|---|
| 🎤 Microphone → audio array | **L32-L33** | numpy signal representation, sample rate |
| 📝 ASR transcription | **L66-L70** | CTC alignment, Whisper architecture |
| 🧠 LLM retrieval + answer | **L83-L90** | Transformer, RAG, ReAct Agent |

So the hard part of this lesson isn't any single module — it's the **seams**: how latency adds up, and how the chain keeps from collapsing when one link fails. Read on with that lens.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import time
import numpy as np

# ── Prerequisite module check (if any is missing, finish the matching notebook first) ──
import importlib, sys

REQUIRED = {
    "aurora.audio.io":        "L32-L36 (signal basics)",
    "aurora.audio.transforms": "L37-L41 (DFT/FFT)",
    "aurora.audio.mel":        "L46-L47 (mel filters)",
}
missing = []
for mod, lesson in REQUIRED.items():
    try:
        importlib.import_module(mod)
        print(f"✅ {mod}")
    except ImportError:
        print(f"❌ {mod}  →  finish this first: {lesson}")
        missing.append(mod)

if missing:
    print(f"\n⚠  {len(missing)} module(s) not ready; this notebook's import cells will fail.")
    print("   The asyncio concurrency example (Section 4) and the monitoring example (Section 5) run on their own and are unaffected.")
else:
    print("\n✅ All dependencies ready; every cell can run.")

# read_wav is ready (aurora.audio.io); this notebook simulates audio input with np.random;
# to use a real recording instead: audio, sr = read_wav("your_recording.wav")

## 1. Pipeline latency breakdown

End-to-end latency is four serial durations added together:

```
total latency = T_record + T_asr + T_rag + T_llm
              ≈    0ms   + 500ms +  50ms + 2000ms
              ≈ 2550ms
```

Here `T_record` is how long the user speaks (usually 1-5 seconds, but the recording itself doesn't count as latency — the user is already talking). Perceived latency starts the moment the user **stops speaking**.
`T_asr` depends on audio length and model size; `T_rag` depends on index size; `T_llm` is the main bottleneck.

Ways to optimize: run ASR and RAG concurrently (`asyncio`), and stream the LLM's output (cutting the **wait for the first token**).

In [ ]:
# Simulated duration of each stage (seconds)
stages = {
    'record':  0.0,    # the recording itself isn't waiting time
    'asr':     0.50,
    'rag':     0.05,
    'llm':     2.00,
}

total = sum(stages.values())
print(f'Duration per stage: {stages}')
print(f'End-to-end latency: {total*1000:.0f}ms')

# Visualize each stage's share of the time
import matplotlib.pyplot as plt
labels = list(stages.keys())
values = list(stages.values())
fig, ax = plt.subplots(figsize=(7, 2))
left = 0
colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
for label, val, color in zip(labels, values, colors):
    if val > 0:
        ax.barh(0, val, left=left, color=color, edgecolor='white', height=0.4, label=label)
        ax.text(left + val/2, 0, f'{label}\n{val*1000:.0f}ms',
                ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        left += val
ax.set_xlim(0, total * 1.05)
ax.set_yticks([])
ax.set_xlabel('Time (s)')
ax.set_title('End-to-end latency breakdown (perceived latency starts when the user stops speaking)')
plt.tight_layout()
plt.savefig('pipeline_latency.png', dpi=120)
plt.show()
print('✅ Latency breakdown chart saved')

## 2. Error handling and fallbacks

Each stage has its own failure modes:

| Stage | Common failure | Fallback |
|------|----------|----------|
| Recording | silence / noise | detect RMS < threshold, ask the user to repeat |
| ASR | empty transcript | return an empty string, skip the rest |
| RAG | no matching document | if similarity < threshold, return a generic answer |
| LLM | timeout / OOM | set `timeout=5s`, return a cached answer on failure |

The key design principle: **every stage returns a `(result, ok, error_msg)` triple**, and the layer above checks `ok` to decide whether to continue. Don't use exceptions to steer the pipeline — an exception is a program bug, while an empty recording is perfectly normal input.

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class StageResult:
    value: Any
    ok: bool
    error: str = ''


def check_audio(audio: np.ndarray, sr: int = 16000,
                rms_threshold: float = 0.01) -> StageResult:
    """Recording quality check: RMS below the threshold counts as silence."""
    rms = np.sqrt(np.mean(audio ** 2))
    if rms < rms_threshold:
        return StageResult(None, False, f'silence (RMS={rms:.4f} < {rms_threshold})')
    return StageResult(audio, True)


# Demo: normal audio vs silence
audio_ok  = np.random.randn(16000) * 0.1   # normal speech
audio_sil = np.zeros(16000)                 # complete silence

r1 = check_audio(audio_ok)
r2 = check_audio(audio_sil)
print(f'Normal audio: ok={r1.ok}')
print(f'Silent audio: ok={r2.ok}, error="{r2.error}"')
assert r1.ok and not r2.ok
print('✅ Silence detection works')

## 3. Asynchronous streaming output

LLM generation usually takes 2-5 seconds, but the **first token** often arrives within 200ms. Streaming lets the user see the start of the answer right away, cutting perceived latency from 2s to 0.2s.

How it works: the LLM returns a generator that `yield`s one token at a time, and the caller prints as it consumes:

```python
for token in llm.stream(prompt):
    print(token, end='', flush=True)
```

In Aurora, the streaming interface is planned as `aurora.llm.agent.stream_answer(query, context)` (a **planned module** — `aurora.llm` has no `agent` submodule yet).
Note: the streaming and batch interfaces produce **exactly the same final output**; the only difference is when it becomes visible.

In [ ]:
import time

def mock_llm_stream(prompt: str, tokens_per_sec: float = 20.0):
    """Simulate LLM streaming output, with 1/tokens_per_sec seconds between tokens."""
    response = 'The Fourier transform splits a signal into its frequencies.'
    tokens = response  # yield character by character to simulate tokens
    for char in tokens:
        time.sleep(1.0 / tokens_per_sec)
        yield char


# Demo of streaming output
print('LLM streaming output (simulated at 20 tokens/s):')
print('─' * 40)
t0 = time.time()
full_response = []
for token in mock_llm_stream('What is the Fourier transform?'):
    print(token, end='', flush=True)
    full_response.append(token)
print()
print('─' * 40)
elapsed = time.time() - t0
print(f'Total time: {elapsed:.2f}s, {len(full_response)} characters')
print('✅ Streaming demo complete')

## 4. asyncio concurrency: turning serial ASR+RAG into parallel

ASR and RAG have no data dependency — while ASR is transcribing, RAG can be loading its index or warming up a query at the same time.
Running the two in parallel with `asyncio.gather()` cuts perceived latency from `2550ms → 2530ms` (saving 20ms); after the parallel ASR+RAG step there is an extra RAG refinement query (30ms), so the overall saving is modest;
if RAG grows to 200ms in the future, the saving becomes much more worthwhile.

```
serial:   [─── ASR 500ms ───][─ RAG 50ms ─][─── LLM 2000ms ───]  = 2550ms
parallel: [─── ASR 500ms ───────────────]
          [─ RAG 50ms ─]                                          = 2530ms (-20ms)
                        [─ RAG refine 30ms ─]
                                  [─── LLM 2000ms ───]
```

**HuggingFace Whisper integration** (replacing the simulated ASR):

```python
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="openai/whisper-small",
               device="cpu", return_timestamps=False)

def run_asr_hf(audio_array: np.ndarray, sr: int = 16000) -> str:
    result = asr({"sampling_rate": sr, "raw": audio_array})
    return result["text"].strip()
```

On CPU, Whisper-small needs about 3-6s for 10s of audio; Whisper-tiny about 1-2s (with a slightly higher WER).

In [ ]:
import asyncio
import time
import numpy as np

# ── Async version of the three-stage pipeline ─────────────────────────────────

async def async_asr(audio: np.ndarray, delay: float = 0.5) -> str:
    """Simulated Whisper ASR (in practice, replace with a HuggingFace pipeline call)"""
    await asyncio.sleep(delay)
    return "What are mel-frequency cepstral coefficients?"

async def async_rag(query: str = None, delay: float = 0.05) -> str:
    """Simulated TF-IDF RAG retrieval (query may be None during the index warm-up)"""
    await asyncio.sleep(delay)
    return "MFCCs extract acoustic features using a mel filterbank and a DCT..."

async def async_llm(transcript: str, context: str, delay: float = 2.0) -> str:
    """Simulated LLM generation (in practice, token-by-token output via streaming + asyncio.Queue)"""
    await asyncio.sleep(delay)
    return f'[Answer] The answer to "{transcript[:20]}..." is based on: "{context[:30]}..."'

async def run_pipeline_parallel(audio: np.ndarray) -> dict:
    """ASR and RAG run in parallel; the LLM waits serially for both to finish."""
    t_start = time.perf_counter()

    # ASR + RAG concurrently
    # Here context acts as an "index warm-up" — in a real system, the first RAG call triggers index loading/caching,
    # bringing the later rag_refine latency down from ~200ms to ~30ms; in the mock it has no real effect and only shows the pattern.
    transcript, context = await asyncio.gather(
        async_asr(audio),
        async_rag(),          # the RAG warm-up doesn't depend on the ASR result
    )
    t_after_asr_rag = time.perf_counter()

    # Precise RAG query (depends on the ASR result)
    context_refined = await async_rag(transcript, delay=0.03)
    t_after_refine = time.perf_counter()

    # LLM generation
    answer = await async_llm(transcript, context_refined)
    t_end = time.perf_counter()

    return {
        "transcript": transcript,
        "answer": answer,
        "timings": {
            "asr+rag_parallel": (t_after_asr_rag - t_start) * 1000,
            "rag_refine":       (t_after_refine - t_after_asr_rag) * 1000,
            "llm":              (t_end - t_after_refine) * 1000,
            "total":            (t_end - t_start) * 1000,
        }
    }

async def run_pipeline_serial(audio: np.ndarray) -> dict:
    """Serial version (control group)"""
    t_start = time.perf_counter()
    transcript = await async_asr(audio)
    context    = await async_rag(transcript)
    answer     = await async_llm(transcript, context)
    total_ms   = (time.perf_counter() - t_start) * 1000
    return {"transcript": transcript, "answer": answer, "timings": {"total": total_ms}}

# ── Run the comparison ───────────────────────────────────────────────────────
audio_sample = np.random.randn(16000) * 0.1

print("=== Serial vs parallel pipeline ===\n")
result_serial   = await run_pipeline_serial(audio_sample)
result_parallel = await run_pipeline_parallel(audio_sample)

print(f"Serial total latency   : {result_serial['timings']['total']:.0f}ms")
print(f"Parallel total latency : {result_parallel['timings']['total']:.0f}ms")
speedup = result_serial['timings']['total'] / result_parallel['timings']['total']
print(f"Speedup                : {speedup:.2f}×\n")

print("Parallel stages:")
for stage, ms in result_parallel['timings'].items():
    print(f"  {stage:20s}: {ms:.0f}ms")

print(f"\nTranscript: {result_parallel['transcript']}")
print(f"Answer excerpt: {result_parallel['answer'][:60]}...")

# ── Template for a real HuggingFace Whisper integration ───────────────────────
HF_WHISPER_TEMPLATE = '''
# Replace async_asr() with a real HuggingFace Whisper:
from transformers import pipeline as hf_pipeline
import numpy as np

_asr_pipe = None

def get_asr_pipeline(model: str = "openai/whisper-small"):
    global _asr_pipe
    if _asr_pipe is None:
        _asr_pipe = hf_pipeline(
            "automatic-speech-recognition",
            model=model,
            device="cpu",                     # GPU: device=0
            return_timestamps=False,
            generate_kwargs={"language": "en"},  # English inference
        )
    return _asr_pipe

async def async_asr_real(audio: np.ndarray, sr: int = 16000) -> str:
    # asyncio.to_thread wraps the synchronous HF call in a coroutine
    pipe = get_asr_pipeline()
    result = await asyncio.to_thread(
        pipe, {"sampling_rate": sr, "raw": audio.astype(np.float32)}
    )
    return result["text"].strip()
'''
print("\n=== HuggingFace Whisper integration template ===")
print(HF_WHISPER_TEMPLATE)

## Parameter experiment: end-to-end timing

Run five spoken question-and-answer rounds (simulated here), measure and record the duration of each stage, and identify the slowest bottleneck.

**Parameters**:
- `n_trials = 5`: number of rounds; more rounds give steadier results
- `rms_threshold = 0.01`: silence threshold; raising it rejects more low-volume input
- `asr_timeout = 3.0`: ASR timeout (seconds); past it, fall back to an empty string *(a design target — `simulate_pipeline` below doesn't implement it yet)*
- `llm_timeout = 5.0`: LLM timeout (seconds); past it, return a cached answer *(likewise not implemented yet)*

**What to expect**: the LLM takes 70-80% of the total latency; with streaming on, perceived latency drops noticeably but the total time stays the same.

In [ ]:
import random

def simulate_pipeline(rms_threshold=0.01, asr_delay=0.5,
                      rag_delay=0.05, llm_delay=2.0):
    """Simulate the full pipeline and return a dict of per-stage durations."""
    timings = {}

    # Stage 1: recording quality check
    t0 = time.perf_counter()
    audio = np.random.randn(16000) * 0.1
    result = check_audio(audio, rms_threshold=rms_threshold)
    timings['audio_check'] = time.perf_counter() - t0
    if not result.ok:
        return timings, result.error

    # Stage 2: ASR transcription (simulated)
    t0 = time.perf_counter()
    time.sleep(asr_delay + random.uniform(-0.05, 0.05))
    transcript = 'What are mel-frequency cepstral coefficients?'
    timings['asr'] = time.perf_counter() - t0

    # Stage 3: RAG retrieval (simulated)
    t0 = time.perf_counter()
    time.sleep(rag_delay + random.uniform(-0.01, 0.01))
    context = 'MFCCs extract acoustic features using a mel filterbank and a DCT...'
    timings['rag'] = time.perf_counter() - t0

    # Stage 4: LLM generation (simulated)
    t0 = time.perf_counter()
    time.sleep(llm_delay + random.uniform(-0.1, 0.1))
    timings['llm'] = time.perf_counter() - t0

    return timings, 'ok'


# Run 5 times and watch how each stage's duration fluctuates
n_trials = 5
all_timings = []

print(f'Running the end-to-end pipeline {n_trials} times...')
for i in range(n_trials):
    timings, status = simulate_pipeline()
    total = sum(timings.values())
    all_timings.append(timings)
    print(f'  Trial {i+1}: asr={timings["asr"]*1000:.0f}ms  '
          f'rag={timings["rag"]*1000:.0f}ms  '
          f'llm={timings["llm"]*1000:.0f}ms  '
          f'total={total*1000:.0f}ms')

# Find the bottleneck
avg = {k: np.mean([t[k] for t in all_timings]) for k in all_timings[0]}
bottleneck = max(avg, key=avg.get)
print(f'\nAverage duration per stage: { {k: f"{v*1000:.0f}ms" for k, v in avg.items()} }')
print(f'Slowest bottleneck: {bottleneck} ({avg[bottleneck]*1000:.0f}ms)')
assert bottleneck == 'llm', 'Expected the LLM to be the main bottleneck'
print('✅ Bottleneck analysis complete')

## 5. Pipeline observability: logs, metrics and alerts

Before the assembled pipeline goes live, it needs three layers of visibility:

| Layer | Example tools | What it monitors |
|---|---|---|
| Structured logs | `structlog` / `logging` | duration and error code of every frame / request |
| Metrics | Prometheus + Grafana | throughput, P50/P95 latency, error rate |
| Alerts | Slack webhook | automatic notification when RTF > 1.0 |

The minimum viable observability (for Aurora): time every stage with `time.perf_counter()`,
emit structured JSON logs, and analyze them after launch with `grep` or ELK.

In [ ]:
import time, logging, json

logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger("aurora")

def timed_stage(name, fn, *args):
    t0 = time.perf_counter()
    result = fn(*args)
    ms = (time.perf_counter() - t0) * 1000
    logger.info(json.dumps({"stage": name, "latency_ms": round(ms, 2)}))
    return result

timed_stage("audio_load",   lambda: time.sleep(0.01) or b"pcm")
timed_stage("asr",          lambda: time.sleep(0.05) or "hello")
timed_stage("llm_generate", lambda: time.sleep(0.08) or "world")
print("Pipeline finished; structured logs emitted")

## 6. ✏️ Exercise: find your pipeline's bottleneck

Fill in the table with latencies measured on your own machine, find the stage that contributes most to P95, and write down one idea for improving it.

In [ ]:
# Section 6 exercise: measure each stage's latency automatically with simulate_pipeline() and identify the bottleneck
# simulate_pipeline() is defined in the parameter-experiment cell above; just run the code below, nothing to fill in by hand.

import numpy as np

# ── Run 5 times and average (adjust n_trials as you like) ─────────────────────
n_trials = 5
all_runs = []
for _ in range(n_trials):
    timings, status = simulate_pipeline()
    if status == 'ok':
        all_runs.append(timings)

if not all_runs:
    print("⚠  Every trial failed the silence check; lower rms_threshold and try again.")
else:
    stage_latency_ms = {k: np.mean([r[k] for r in all_runs]) * 1000
                        for k in all_runs[0]}
    total = sum(stage_latency_ms.values())
    bottleneck = max(stage_latency_ms, key=stage_latency_ms.get)

    print(f"Average total latency: {total:.1f} ms   Bottleneck stage: {bottleneck}")
    print()
    for k, v in sorted(stage_latency_ms.items(), key=lambda x: -x[1]):
        bar = chr(9608) * int(v / total * 30)
        pct = v / total * 100
        print(f"  {k:<20} {v:>6.1f}ms  {pct:4.1f}%  {bar}")

    # ── Check ─────────────────────────────────────────────────────────────────
    assert bottleneck == 'llm', (
        f"Expected the LLM to be the main bottleneck, but it was {bottleneck}. "
        f"Check whether the delay parameters of simulate_pipeline have been changed."
    )
    print(f"\n✅ Bottleneck check passed: {bottleneck} accounts for {stage_latency_ms[bottleneck]/total*100:.1f}%")

    # ✏️ Thought question: if llm_delay drops from 2.0s to 0.5s, by what percentage does total latency fall?
    # (Hint: run simulate_pipeline(llm_delay=0.5) again and compare)


## Wrap-up

`check_audio` guards the quality of what comes in, and the `StageResult` triple makes every stage's failure observable; the four serial durations add up to the end-to-end latency, and LLM generation takes about 80% of it, which makes it the first thing to optimize. This lesson used mock functions as placeholders to demonstrate the end-to-end integration pattern: it called `aurora.audio.io` (recording read/write) and showed the hook-up interfaces for `aurora.llm.retrieve` / `aurora.llm.sample`; `aurora.speech` (ASR) and `aurora.rag` (retrieval) are planned — when they arrive, swap out the matching mock functions and the loop closes. Section 4 used `asyncio` concurrency to run ASR and RAG in parallel and bring down perceived latency. Next lesson: **L93** MLOps basics — experiment tracking with W&B, model versioning, and Docker packaging.

---

→ **Next lesson** · [L93 · MLOps basics](L93_mlops.ipynb)

> Next lesson covers **MLOps basics**: experiment tracking with W&B, model versioning, Docker packaging and deployment scripts.